# Two-signal experiments — Colab GPU runbook (S0→S4)

Runs the length (Signal L) + error-tag (Signal E) matrix on a GPU and produces the
distance-to-learner figures/tables. Reuses the existing batch pipeline + registry (no rewrite).
**Runtime → Change runtime type → GPU** first. Run cells top to bottom.
Vocabulary is canonical (CONCEPTS.md): authentic learner / matched control / artificial learner (AL).

S0 smoke = 1 pair × CELVA (cap 20) × 1 seed (proves the path). S1–S4 scale up in Cell 4.

## Cell 1 — GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), 'Set Runtime -> Change runtime type -> GPU'
print('GPU:', torch.cuda.get_device_name(0))

## Cell 2 — Mount Google Drive (guarded so a non-Colab dry-run skips it)

In [ ]:
# ── ENV DETECT (branch-on-env ONCE, fail-loud) ──
# Detect Colab by google.colab IMPORTING (the real signal) — NOT by drive.mount succeeding.
# A transient mount error must never be miscaught as "not on Colab" and silently take the
# local install branch (which skips installing gen_gec_errant → import-error cascade).
# The mount is a SEPARATE inner step that fails LOUD and NEVER flips IN_COLAB.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print('FATAL: on Colab but Drive mount failed (%s: %s).' % (type(e).__name__, e))
        print('Re-auth / retry the mount and re-run — NOT falling back to a local install.')
        raise
    print('Colab detected; Drive mounted.')
else:
    print('Not on Colab (google.colab import failed) — local/Jupyter run, Drive mount skipped.')
print('IN_COLAB:', IN_COLAB)


## Cell 3 — Get the code (PUBLIC mirror, with a Drive `_code` fallback) + install
Never a **private** clone on a cloud runtime (that killed the first run). Code comes from the
scrubbed **public mirror** `gen-gec-errant-public` — no data/models/secrets in it — or, as a
guaranteed-current fallback, the `_code` tree copied onto the Drive the runtime already mounts.
Install is idempotent subprocess pip (no `!pip`); `sys.path.insert` makes the editable install
importable in-process.

In [ ]:
import subprocess, sys, os, shutil
# Code acquisition — NEVER a private clone on a cloud runtime (auth the VM lacks).
#   (a) PUBLIC mirror   — scrubbed, no data/models/secrets in it          [default]
#   (b) copied _code tree on the Drive the runtime already mounts         [fallback]
PUBLIC_REPO_URL = 'https://github.com/berstearns/gen-gec-errant-public.git'
DRIVE_CODE      = '/content/drive/MyDrive/_p/artificial-learners/generation-gec-errant/_code'
REPO            = '/content/gen-gec-errant'
USE_DRIVE_CODE  = False   # True → force the guaranteed-current Drive copy instead of the public clone

if IN_COLAB:
    if not os.path.exists(REPO):
        if USE_DRIVE_CODE:
            print('code: copying _code tree from Drive (forced fallback)…')
            shutil.copytree(DRIVE_CODE, REPO)
        else:
            try:
                subprocess.run(['git', 'clone', '--depth', '1', PUBLIC_REPO_URL, REPO], check=True)
            except subprocess.CalledProcessError as e:
                print(f'public clone failed ({e}); falling back to the Drive _code tree…')
                shutil.copytree(DRIVE_CODE, REPO)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO])
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
    os.chdir(REPO)
else:
    REPO = os.getcwd()   # local dry-run: the working tree IS the repo
sys.path.insert(0, os.path.join(REPO, 'src'))   # editable install: .pth is read only at interpreter startup
print('repo:', REPO, '| cwd:', os.getcwd())


## Cell 4 — Configure the run (the ONE place you edit)
S0 smoke defaults below. For S1: add pairs, set `SEEDS=[42,43,44]`, drop `MAX_SENTENCES`.
`BROKER` chooses HOW inputs are acquired — `gdrive` (mounted Drive, the easy route) · `rclone`
(any remote + delivered conf) · `fserve` (authenticated curl, zero-config) · `local` · `hf`.

In [ ]:
from pathlib import Path
from gen_gec_errant.registry import MODEL_REGISTRY, DATASET_REGISTRY, PathConfig, build_pipeline_config
from gen_gec_errant.colab import is_colab
from gen_gec_errant.nbenv import layered_get   # env var > .env (repo root) > Colab userdata > default (fail-loud)

PAIRS = [   # matched pairs: (AL registry key, matched-control registry key)
    ('ft-gpt2-small', 'gpt2-small-native'),
    # ('ft-gpt2-medium','gpt2-medium-native'), ('ft-pythia-410m','pythia-410m-native'), ...
]
DATASET       = 'norm-CELVA-SP'   # contamination-clean cross-corpus (primary)
SEEDS         = [42]              # S0 = 1 seed; S1 >= 3 seeds (single-seed is a known threat)
MAX_SENTENCES = 20               # S0 cap; set None for the full corpus (S1+)
DECODE        = {'max_new_tokens': 64, 'min_new_tokens': 1, 'repetition_penalty': 1.2}  # ratified 2026-07-15

paths = PathConfig.for_colab() if is_colab() else PathConfig.for_local()
# TEMPLATED Drive layout — public users point at their OWN MyDrive layout via .env / env vars
# (defaults keep the standard layout; nothing machine-specific is hardcoded).
paths = PathConfig(
    data_root   = Path(layered_get('GGE_DRIVE_DATA_ROOT',   str(paths.data_root))),
    models_root = Path(layered_get('GGE_DRIVE_MODELS_ROOT', str(paths.models_root))),
    output_root = Path(layered_get('GGE_DRIVE_OUTPUT_ROOT', str(paths.output_root))),
)

# ── acquisition backend — TEMPLATED: no private value is hardcoded here ──
# See .env.template for every key; a required-but-unset key fails loud naming itself.
BROKER = layered_get('GGE_BROKER', 'gdrive')   # gdrive | rclone | fserve | local | hf
BROKER_SETTINGS = {
    'gdrive': dict(paths=paths),
    'local':  dict(paths=paths),
    'rclone': dict(remote=layered_get('GGE_RCLONE_REMOTE', 'PLACEHOLDER_RCLONE_REMOTE', required=(BROKER == 'rclone')),
                   conf_path=layered_get('GGE_RCLONE_CONF_PATH')),                 # conf delivered at runtime, never committed
    'fserve': dict(base_url=layered_get('GGE_FSERVE_BASE_URL', 'PLACEHOLDER_FSERVE_BASE_URL', required=(BROKER == 'fserve')),
                   user=layered_get('GGE_FSERVE_USER', 'colab'),
                   passcode=layered_get('GGE_FSERVE_PASSCODE', required=(BROKER == 'fserve')),  # prefer Colab userdata
                   codename_map={}),                                               # fill from `do-fserve-ls`: logical name -> codename
    'hf':     dict(),
}[BROKER]
print('is_colab:', is_colab(), '| broker:', BROKER, '| dataset:', DATASET, '| pairs:', PAIRS, '| seeds:', SEEDS)


## Cell 4b — Acquire inputs through the BROKER
**Input:** `BROKER` + `BROKER_SETTINGS` + the run's `PAIRS`/`DATASET`.  **Idea:** resolve logical
resource names (`corpora/…`, `checkpoints/…`) to *local, verified* paths via the chosen backend —
no cell holds a URL; every fetch is idempotent + verified.  **Output:** `LOCAL_PATHS` (logical
name → local Path) consumed by preflight + the run.  Controls are **not** brokered (HF Hub by id).

In [ ]:
from gen_gec_errant import brokers as bk

# resources THIS run needs: the corpus + one AL checkpoint per pair (controls load from HF Hub)
CORPUS_RES = f"corpora/{DATASET_REGISTRY[DATASET].filename}"
CKPT_RES   = {al_key: f"checkpoints/{al_key}" for al_key, _ in PAIRS}

manifest = bk.build_manifest(model_keys=[al for al, _ in PAIRS], dataset_keys=[DATASET])
broker   = bk.make_broker(BROKER, manifest, **BROKER_SETTINGS)

# Credential delivery for rclone/fserve — NEVER commit these; fetch at runtime:
#   from google.colab import userdata                       # channel (a): Colab userdata
#   BROKER_SETTINGS['passcode'] = userdata.get('FSERVE_PASSCODE')
#   # channel (b): bootstrap rclone.conf from an fserve codename, then point RcloneBroker at it:
#   #   subprocess.check_call(['curl','-fsSL','-k','-u',f'colab:{PASS}',
#   #                          'https://DROPLET_IP:47824/rconf','-o','/content/rclone.conf'])
#   #   BROKER_SETTINGS['conf_path'] = '/content/rclone.conf'

BROKER_ROOT = Path('/content/brokered' if is_colab() else '/tmp/gge-brokered')
LOCAL_PATHS = {}   # logical resource name -> local Path (verified by the broker)
LOCAL_PATHS[CORPUS_RES] = broker.acquire(CORPUS_RES, BROKER_ROOT / 'corpora')
for al_key, res in CKPT_RES.items():
    LOCAL_PATHS[res] = broker.acquire(res, BROKER_ROOT / 'checkpoints' / al_key)
print('broker:', broker.name, '| acquired:', {k: str(v) for k, v in LOCAL_PATHS.items()})


## Cell 4c — PREFLIGHT (fail fast, through the broker)
**Input:** `LOCAL_PATHS` + `manifest` + `PAIRS`.  **Idea:** re-verify every brokered input, echo the
controls (HF Hub) + env/GPU, and **raise** on any ❌ — ~5 s of checks before hours of GPU, so the
run can never start on a missing/corrupt input.  **Output:** ✅/❌ per input; RuntimeError if any ❌.

In [ ]:
# ── PREFLIGHT — validate every input THROUGH the broker, fail fast (≈5s) ──
# The BROKER cell acquired+verified into LOCAL_PATHS; here we re-verify those local paths via
# brokers.verify, echo the controls (HF Hub, not brokered) + env, and RAISE on any ❌ so the
# hours-long run cell can never start on a missing/corrupt input. Colab-safe (no !pip, no magics).
fail = False
print(f"── PREFLIGHT (broker '{broker.name}') ────────────────────────")

# brokered resources (corpus + AL checkpoints) — verify the acquired local path
for res, local in LOCAL_PATHS.items():
    if bk.verify(manifest[res], Path(local)):
        print(f"✅ {res}: verified at {local}")
    else:
        print(f"❌ {res}: MISSING/INVALID at {local}")
        fail = True

# matched controls load from the HF Hub (not brokered — nothing to fetch)
for al_key, ctrl_key in PAIRS:
    m = MODEL_REGISTRY[ctrl_key]
    print(f"✅ {ctrl_key}: HF Hub id '{m.hf_model_id}' (loads from Hub)")

# package + environment
try:
    import gen_gec_errant
    print(f"✅ gen_gec_errant {getattr(gen_gec_errant, '__version__', '(no __version__)')}")
except Exception as e:  # noqa: BLE001
    print(f"❌ gen_gec_errant import failed: {e}")
    fail = True

_cuda = torch.cuda.is_available()
print(f"   broker              : {broker.name}")
print(f"   is_colab            : {is_colab()}")
print(f"   torch.cuda available: {_cuda}" + (f" ({torch.cuda.get_device_name(0)})" if _cuda else ""))
print(f"   models_root         : {paths.models_root}")
print(f"   data_root           : {paths.data_root}")
print(f"   output_root         : {paths.output_root}")

if fail:
    print()
    print("!" * 62)
    print("!!  PREFLIGHT FAILED — inputs marked ❌ above are missing.     !!")
    print("!!  Fix the broker config / storage before running the matrix.!!")
    print("!" * 62)
    raise RuntimeError("preflight failed — fix the flagged inputs before running")
print("\n✅ preflight OK — safe to run the matrix.")


## Cell 5 — Run the matrix (GPU, one model at a time, resume-enabled)
Each (model, seed) writes `raw_results.json` + `learner_profile.json` + checkpoints to its own dir.
`resume=True` + Drive output survives a 12h disconnect — re-run this cell and it skips completed steps.

In [ ]:
from gen_gec_errant.pipeline.runner import run_pipeline

CORPUS_LOCAL = str(LOCAL_PATHS[CORPUS_RES])
RUN_DIRS = {}  # (registry_key, seed) -> output_dir
for al_key, ctrl_key in PAIRS:
    for key in (al_key, ctrl_key):
        model = MODEL_REGISTRY[key]
        # AL → brokered local checkpoint; control → None (loads from the HF Hub by id)
        mpath = str(LOCAL_PATHS[f"checkpoints/{key}"]) if model.gdrive_subpath else None
        for seed in SEEDS:
            cfg = build_pipeline_config(model, DATASET_REGISTRY[DATASET], paths, model_path=mpath,
                                        max_sentences=MAX_SENTENCES)
            cfg.data_loader.data_path = CORPUS_LOCAL              # the brokered corpus
            cfg.generation.max_new_tokens = DECODE['max_new_tokens']
            cfg.generation.min_new_tokens = DECODE['min_new_tokens']
            cfg.generation.repetition_penalty = DECODE['repetition_penalty']
            cfg.seed = seed
            cfg.output_dir = f"{paths.output_root}/two-signal/{DATASET}/{key}/seed{seed}"
            cfg.resume = True
            cfg.include_learner_baseline = True                  # the authentic-learner yardstick
            cfg.skip_plots = True                                # our own two_signal figures replace them
            print('RUN', key, 'seed', seed, '->', cfg.output_dir)
            run_pipeline(cfg)
            RUN_DIRS[(key, seed)] = cfg.output_dir
print('done; run dirs:', RUN_DIRS)


## Cell 6 — Compute BOTH signals + the distance plane (post-hoc; Tier 1, no pipeline change)
`scripts/analysis/two_signal.py`: Signal L (length_ratio, over-generation, n_sentences, length_distance)
+ Signal E (gen-region ERRANT JSD-to-learner, RDR, acquisition ratios) -> `matched_pairs.tsv` + `distance_plane.json`.

In [ ]:
import subprocess, sys, os
ANALYSIS = {}  # pair label -> analysis out dir
for al_key, ctrl_key in PAIRS:
    seed = SEEDS[0]
    al_dir, ctrl_dir = RUN_DIRS[(al_key, seed)], RUN_DIRS[(ctrl_key, seed)]
    out = f"{paths.output_root}/two-signal/{DATASET}/_analysis/{al_key}__{ctrl_key}/seed{seed}"
    os.makedirs(out, exist_ok=True)
    subprocess.check_call([sys.executable,'-m','scripts.analysis.two_signal',
                           '--run-dirs', str(al_dir), str(ctrl_dir),
                           '--pair', f'{al_key}:{ctrl_key}', '--out', out])
    ANALYSIS[f'{al_key}:{ctrl_key}'] = out
    print('signals ->', out)
# (S1: >=3 seeds — run per seed and average, or extend two_signal to accept multiple seed dirs.)

## Cell 7 — Figures & tables (Figs 1–4 + Tables 1–2, canonical 3-source palette; PNG + SVG)

In [ ]:
for label, out in ANALYSIS.items():
    subprocess.check_call([sys.executable,'-m','scripts.analysis.two_signal_figures','--in', out])
    print('figures/tables ->', out + '/{figures,tables}')

## Cell 8 — Collect (zip figures/tables + matched_pairs.tsv + distance_plane.json to Drive)

In [ ]:
import shutil, os
for label, out in ANALYSIS.items():
    base = out.rstrip('/')
    zip_path = shutil.make_archive(base + '_bundle', 'zip', out)
    print('bundle:', zip_path)
    for f in ('matched_pairs.tsv','distance_plane.json','signals.json'):
        p = os.path.join(out, f)
        print('  ', p, 'OK' if os.path.exists(p) else 'MISSING')